# Token & Reasoning Transparency

### Where do the tokens actually go — and how much of it can anyone see?

---

## The question

Every agent API tells you a total: *this request used 12,431 tokens.* Almost none tell
you **where they went** — how much was system prompts, tool definitions you never used,
retrieved documents, conversation history re-sent for the tenth time, or reasoning the
model performed and billed you for but never showed you.

This notebook produces that breakdown. Not as a proposal — as a working artifact.

## Three claims, in order of strength

1. **Here is the ceiling.** What is knowable when you control every layer except the
   model itself, demonstrated end-to-end.
2. **Most of the gap is *withheld*, not *impossible*.** Per-call counts, cached tokens,
   and reasoning tokens are already returned by the API. A vendor not surfacing them is
   making a disclosure choice, not hitting a technical limit.
3. **The distance between (1) and what commercial agents disclose is the transparency
   gap** — quantified rather than asserted.

## What is and isn't knowable

| Layer | Why we can get it | Could a vendor disclose it? | Typically? |
|---|---|---|---|
| Per-call token counts | The API already returns it | **Yes, trivially** | Rarely |
| Reasoning / cached tokens | Provider reports it | **Yes** — pure passthrough | Almost never |
| Context composition | We assembled the prompt | Yes, in aggregate | No |
| Inside the model | — | **No — genuinely impossible** | n/a |

Only the last row is a technical limit.

## Honest limits, stated up front

Even at the ceiling we are not omniscient. Every number in this notebook carries a
**tier**:

- **verified** — we counted it ourselves from content we assembled
- **trusted** — the vendor reported it; we cannot audit it
- **asserted** — reported but uninspectable (reasoning tokens)

And a **residual** row is always shown. If our decomposition disagrees with the API's
reported total, that disagreement is printed rather than hidden.

> Full methodology: [`docs/transparency_spec.md`](docs/transparency_spec.md)

## 1. Setup

All implementation lives in `src/`. This notebook is the narrative.

| Module | Role |
|---|---|
| `support_dataset` / `support_tools` | Corpus + **real** tools under a sandbox contract |
| `support_agents` | Supervisor → Planner → Navigator → Validator, per-call instrumented |
| `otel` | `Usage` (incl. cached + reasoning tokens), `LLMCallRecorder` |
| `attribution` | The headline table and derived metrics |
| `reasoning` | Plan–execution divergence, reasoning provenance |

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import pandas as pd
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 60)

from src import (
    Config, HierarchicalTracer, TracingManager,
    load_support_corpus, init_support_tools, workspace_manifest, TOOL_EFFECT_CLASS,
    create_support_mas, run_support_mas, evaluate_support_outcome,
    attribution, reasoning,
    plot_token_attribution, plot_stage_bars, plot_context_growth, plot_trace_tree,
)

Config.setup_dirs()
print('imports OK')

### Model assignment

The **Planner runs a reasoning model on purpose.** Planning is where extended reasoning
is most defensible *and* most invisible — which makes it the exhibit. The Supervisor
makes no LLM call at all: its routing is a dictionary lookup, so orchestration costs
exactly zero tokens. That is a structural finding, not a gap.

In [ ]:
mas_models = {
    'supervisor': 'none (deterministic routing)',
    'planner':    Config.SUPPORT_PLANNER_MODEL,
    'navigator':  Config.SUPPORT_NAVIGATOR_MODEL,
    'validator':  Config.SUPPORT_VALIDATOR_MODEL,
}
for role, m in mas_models.items():
    tag = ' ← reasoning model' if Config.is_reasoning_model(str(m)) else ''
    print(f'  {role:11} {m}{tag}')

## 2. The support desk — real tools, contained effects

The Mind2Web notebook mocks every WRITE action, so consequences are unobservable and
only *trajectory* can be scored. Here the tools are real:

| Tool | Real action | Effect |
|---|---|---|
| `search_kb`, `read_policy`, `read_order`, `read_ticket` | real file reads | read / local |
| `web_search` | real Tavily (optional) | read / **external** |
| `draft_response` | **real file write** to `workspace/drafts/` | **write / local** |
| `escalate` | **real append** to `workspace/escalations.jsonl` | **write / local** |

The blast radius is contained by a contract enforced in code, not convention:
a path-traversal guard on every write, no network writes, a per-run allowlist, safety
validation on write-tool **inputs**, and a gitignored workspace reset each run.

### The corpus is a measuring instrument

It is synthetic on purpose — like a resolution chart for a camera lens. You don't shoot
a test chart because you can't find real scenes; you shoot it because its properties are
**known**, which is what makes the measurement falsifiable. Each planted trap validates
a specific metric, and because there is ground truth we can score **outcomes**, not just
trajectories.

In [ ]:
corpus = load_support_corpus()
workspace = init_support_tools(corpus, reset=True)

s = corpus.summary()
print(f"KB articles {s['kb_articles']} · policies {s['policies']} · orders {s['orders']} · tickets {s['tickets']}")
print(f"difficulty: {s['difficulty']}")
print(f"planted traps: {s['traps']}")
print(f"sandbox workspace: {workspace}")

pd.DataFrame([{
    'id': t.id, 'difficulty': t.difficulty, 'trap': t.trap or '—',
    'subject': t.subject, 'expects': ', '.join(t.expected_articles),
    'escalate': t.should_escalate,
} for t in corpus.tickets])

## 3. Run the multi-agent system

Every LLM call is recorded individually. That matters: aggregate accounting collapses a
multi-turn navigator into one number, which makes re-planning, retries, and per-turn
context growth **unobservable by construction**.

Set `N_TICKETS` lower for a quick pass.

In [ ]:
N_TICKETS = len(corpus.tickets)      # lower this for a faster/cheaper run

mas = create_support_mas(Config)
tracer, tm = HierarchicalTracer(), TracingManager()
print(f'tool definitions cost {mas["tool_schema_tokens"]} tokens — resent on EVERY navigator turn\n')

runs, outcomes = [], []
for t in corpus.tickets[:N_TICKETS]:
    r = run_support_mas(t, mas, tracer, tm, Config)
    o = evaluate_support_outcome(t, r.cited_articles, r.escalated)
    runs.append(r); outcomes.append(o)
    u = r.total_usage()
    flag = '✅' if o.passed else '❌'
    print(f'{flag} {t.id} [{t.difficulty:6}] {len(r.all_calls)} calls · {len(r.tool_calls)} tools · '
          f'{u.total_tokens:6,} tok (hidden {u.reasoning_tokens:4,}) · ${r.total_cost:.4f} · {r.latency_ms:,.0f}ms')

print(f'\nerrors: {sum(len(r.errors) for r in runs)}')
print('workspace:', workspace_manifest()['draft_count'], 'drafts,',
      workspace_manifest()['escalation_count'], 'escalations — real files on disk')

## 4. ⭐ Where the tokens went

**The headline artifact.** One invariant governs this table:

> **Total = Σ over LLM calls of (input + output).** Every row is a *decomposition* of
> that total, never an addition to it.

That rule matters. Tool outputs are tempting to list as their own line, but they are
already *inside* the next call's input tokens — counting them separately would inflate
the denominator and make every share wrong. So input decomposes into context categories
plus a residual; output decomposes into visible text and hidden reasoning.

Every row carries **Method** (how the number was obtained) and **Tier** (how far it can
be trusted).

In [ ]:
attribution.attribution_dataframe(tracer)

In [ ]:
att = attribution.token_attribution(tracer)
print(f"total {att['total_tokens']:,} tokens across {att['llm_calls']} LLM calls")
print(f"input {att['input_tokens']:,} · output {att['output_tokens']:,}")
print(f"hidden reasoning {att['reasoning_tokens']:,} ({att['hidden_share']:.0%} of output)")
print(f"cached input {att['cached_tokens']:,}")
print(f"\nresidual {att['residual_tokens']:+,} ({att['residual_share']:.1%} of input)")
print('  A non-zero residual means our decomposition disagrees with the API total.')
print('  Sources: tokenizer mismatch, per-message framing, tool-schema serialisation.')
print('  It is reported rather than tuned away — that is the honesty mechanism.')

In [ ]:
fig = plot_token_attribution(tracer, save_path=Config.OUTPUT_DIR / 'token_attribution.png')

### The same total, cut by workflow stage

An orthogonal view. Both cuts sum to 100% of the measured total; neither is added to the
other. Note the Supervisor row: **zero tokens, by construction.**

In [ ]:
stages = attribution.stage_breakdown(tracer)
pd.DataFrame([{
    'Stage': s['stage'], 'Tokens': f"{s['tokens']:,}", 'Share': f"{s['share']:.1%}",
    'LLM calls': s['calls'], 'Hidden reasoning': s['hidden_reasoning'],
    'Method': s['method'], 'Tier': s['tier'],
} for s in stages])

## 5. Exhibits

Five findings that standard token accounting cannot surface.

### 5a. The tool-definition tax

Tool schemas are sent with **every single turn**, whether or not the agent calls those
tools. This is invisible in any per-request total, and it is directly actionable: trim
the tool set for a task class and the saving is immediate.

In [ ]:
ts = attribution.tool_schema_overhead(tracer)
print(f"tool definitions: {ts['tokens_per_turn']:,} tokens, resent on each of {ts['turns_charged']} turns")
print(f"total: {ts['total_tokens']:,} tokens = {ts['share_of_input']:.0%} of ALL input")
print(f"distinct tools actually used across the run: {ts['distinct_tools_used']} of {len(mas['tools'])}")
print(f"\nfor comparison, the navigator system prompt is a few dozen tokens.")

### 5b. Context growth — you pay for the same words repeatedly

Every ReAct turn resends the full conversation. New information arrives linearly;
re-sent context accumulates. In long runs this dominates everything else.

In [ ]:
g = attribution.context_growth(tracer)
print(f"navigator input: {g['first_input_tokens']:,} → {g['last_input_tokens']:,} tokens "
      f"({g['growth_factor']:.1f}x)")
print(f"re-sent context (history + prior tool output): {g['resent_share']:.0%} of all navigator input")
fig = plot_context_growth(tracer, save_path=Config.OUTPUT_DIR / 'context_growth.png')

### 5c. Hidden reasoning — billed, but unreadable

The sharpest exhibit. Reasoning models generate a chain of thought, **charge you for
it**, and never return it. The size is reported; the content is withheld. This is a
disclosure choice, not a technical limit.

In [ ]:
h = attribution.hidden_reasoning(tracer)
print(f"hidden reasoning: {h['hidden_tokens']:,} of {h['output_tokens']:,} output tokens "
      f"({h['hidden_share']:.0%})\n")
for agent, d in h['by_agent'].items():
    if d['output']:
        print(f"  {agent:11} {d['reasoning']:5,} / {d['output']:5,} output = {d['hidden_share']:.0%} unreadable")
print('\nYou paid for every one of those tokens. You cannot read any of them.')

### 5d. Cost is not reproducible, even when tokens are

Token counts are deterministic — the same messages always tokenize identically. **Cost
is not.** Cache hits depend on TTL and eviction you do not control, so the same request
billed twice can differ. Attribution *by token* is reproducible; attribution *by dollar*
is an estimate, and this notebook labels it as one.

The cell below demonstrates it directly: one identical prompt, sent twice.

In [ ]:
from langchain_core.messages import HumanMessage
probe_llm = Config.create_llm(role='agent', model=Config.SUPPORT_NAVIGATOR_MODEL)
big = 'Knowledge base article. ' + ('Policy detail sentence for refunds and returns. ' * 260)

for n in (1, 2):
    r = probe_llm.invoke([HumanMessage(content=big + '\n\nReply with the single word OK.')])
    um = r.usage_metadata
    cached = (um.get('input_token_details') or {}).get('cache_read', 0)
    print(f"call {n}: input={um['input_tokens']:,} cached={cached:,} "
          f"({cached/max(um['input_tokens'],1):.0%} served from cache)")
print('\nIdentical tokens. Different cost. Determined by state we do not control.')

### 5e. Duplicate retrievals

Argument fingerprints make repeated identical retrievals visible — did the agent pay
twice for the same information? Costs nothing to detect.

In [ ]:
d = attribution.duplicate_retrievals(tracer)
print(f"tool calls {d['total_calls']} · unique {d['unique_calls']} · duplicates {d['duplicate_calls']}")
print(f"estimated wasted tokens: {d['wasted_tokens_est']:,}")
if d['detail']:
    display(pd.DataFrame(d['detail']))
else:
    print('\nNo duplicates in this run. The metric reports zero honestly — the corpus')
    print('contains bait for duplication, but bait is an opportunity, not a guarantee.')

rp = attribution.replanning_count(tracer)
print(f"\nre-plans: {rp['replans']} · navigator turns: {rp['navigator_turns']} · "
      f"tool calls: {rp['tool_calls']} ({rp['tools_per_turn']:.1f} per turn)")
print('Turns ≠ tool calls: models emit several tool calls per turn (parallel calling).')

## 6. Reasoning transparency

### Architecture is a transparency choice

A single reasoning-model call hides its planning inside the opaque layer. Decomposing
into Planner → Navigator → Validator moves that same reasoning into **inspectable
inter-agent messages**. The multi-agent system costs more *and* makes reasoning
auditable — a dimension the usual quality/cost comparison misses entirely.

### But a reasoning narrative is a *claim about* process, not process

Chain-of-thought is frequently unfaithful: models reach answers via factors they don't
mention, then rationalize. Fluent reasoning is strong evidence of a well-trained writer,
not of a sound process.

So we evaluate reasoning by its **consequences and its consistency with the trace**, not
by reading it.

In [ ]:
inv = reasoning.reasoning_inventory(tracer)
pd.DataFrame(inv.as_rows())

### Plan–execution divergence — the deterministic faithfulness check

We have the Planner's *stated* steps and the Navigator's *actual* tool calls. Comparing
them needs no judge: the plan is a testable claim and the trace is ground truth about
what happened.

This is uniquely enabled by the multi-agent architecture. A single agent that plans
internally leaves nothing to compare against.

In [ ]:
div_rows = []
for r in runs:
    d = reasoning.plan_execution_divergence(r.plan, source={r.trace_id: tracer.traces.get(r.trace_id, [])})
    div_rows.append({
        'ticket': r.ticket_id, 'planned': len(d.planned), 'executed': len(set(d.executed)),
        'followed': len(d.followed), 'skipped': ', '.join(d.skipped) or '—',
        'unplanned': ', '.join(d.unplanned) or '—',
        'fidelity': f'{d.fidelity:.0%}', 'verdict': d.verdict,
    })
pd.DataFrame(div_rows)

### Did the reasoning buy anything?

Correlational, not causal — but a flat or negative relationship is strong evidence that
reasoning spend is decorative. Establishing causality needs ablation (remove the plan,
rerun), which is deliberately out of scope here because it costs a full re-run per
condition.

In [ ]:
records = []
for r, o in zip(runs, outcomes):
    t = next(x for x in corpus.tickets if x.id == r.ticket_id)
    u = r.total_usage()
    records.append({'difficulty': t.difficulty, 'reasoning_tokens': u.reasoning_tokens,
                    'total_tokens': u.total_tokens, 'passed': o.passed})

rc = attribution.reasoning_vs_complexity(records)
print('reasoning spend by task difficulty:')
for diff, v in rc['by_difficulty'].items():
    pr = f"{v['pass_rate']:.0%}" if v['pass_rate'] is not None else 'n/a'
    print(f"  {diff:7} n={v['n']:2}  avg reasoning {v['avg_reasoning_tokens']:7,.0f} tok  "
          f"avg total {v['avg_total_tokens']:7,.0f}  pass {pr}")
print(f"\ncorrelation(difficulty, reasoning): {rc['difficulty_reasoning_correlation']}")
print(f"→ {rc['interpretation']}")

oc = reasoning.reasoning_outcome_correlation(records)
print(f"\ncorrelation(reasoning spend, outcome): {oc['correlation']} (n={oc['n']})")
print(f"→ {oc['interpretation']}")

## 7. Outcome evaluation — beyond trajectory

Because the corpus has ground truth, we can score what actually happened, not just
whether the steps looked sensible.

The distinction matters. An agent that cites the correct article **and** the distractor
article, then escalates when policy says not to, has a perfectly plausible trajectory
— and a wrong outcome. Hedging by citing everything is not a pass.

In [ ]:
pd.DataFrame([{
    'ticket': o.ticket_id, 'difficulty': o.difficulty, 'trap': o.trap or '—',
    'passed': '✅' if o.passed else '❌',
    'cited_correct': o.cited_correct, 'fell_for_trap': o.fell_for_trap,
    'escalation_ok': o.escalation_correct,
    'recall': f'{o.recall:.0%}', 'notes': '; '.join(o.notes) or '—',
} for o in outcomes])

In [ ]:
n = len(outcomes)
print(f"pass rate           {sum(o.passed for o in outcomes)}/{n} = {sum(o.passed for o in outcomes)/n:.0%}")
print(f"cited correctly     {sum(o.cited_correct for o in outcomes)}/{n}")
print(f"fell for a trap     {sum(o.fell_for_trap for o in outcomes)}/{n}")
print(f"escalation correct  {sum(o.escalation_correct for o in outcomes)}/{n}")
by = {}
for o in outcomes:
    by.setdefault(o.difficulty, []).append(o.passed)
print('\nby difficulty: ' + ' · '.join(f'{k} {sum(v)}/{len(v)}' for k, v in by.items()))

## 8. Per-ticket view

In [ ]:
per_task = []
for r in runs:
    sub = {r.trace_id: tracer.traces.get(r.trace_id, [])}
    per_task.append({'label': r.ticket_id,
                     'stages': {s['stage']: s['tokens'] for s in attribution.stage_breakdown(sub)}})
fig = plot_stage_bars(per_task, save_path=Config.OUTPUT_DIR / 'stage_bars.png')

In [ ]:
# Span tree for one ticket — structure, to sit alongside the mass view above
fig = plot_trace_tree(tracer, runs[0].trace_id, save_path=Config.OUTPUT_DIR / 'support_trace_tree.png')

## 9. The transparency gap

Everything above is **the achievable ceiling when you control the stack**. Toggling
components, decomposing context, and instrumenting every call are possible only from
inside. None of it recovers hidden computation in someone else's product.

That boundary is the result, not a caveat. The distance between this ceiling and what a
commercial agent actually discloses **is** the transparency gap.

In [ ]:
ladder = pd.DataFrame([
    {'Disclosure level': 'Total tokens per request',      'We demonstrated': '✅',
     'Vendor could disclose': '✅ trivially', 'Typically disclosed': 'usually'},
    {'Disclosure level': 'Per-call / per-turn tokens',    'We demonstrated': '✅',
     'Vendor could disclose': '✅ trivially', 'Typically disclosed': 'rarely'},
    {'Disclosure level': 'Cached vs fresh input',         'We demonstrated': '✅',
     'Vendor could disclose': '✅ passthrough', 'Typically disclosed': 'almost never'},
    {'Disclosure level': 'Reasoning token count',         'We demonstrated': '✅',
     'Vendor could disclose': '✅ passthrough', 'Typically disclosed': 'almost never'},
    {'Disclosure level': 'Context composition (shares)',  'We demonstrated': '✅',
     'Vendor could disclose': '✅ in aggregate', 'Typically disclosed': 'no'},
    {'Disclosure level': 'Reasoning token content',       'We demonstrated': '❌',
     'Vendor could disclose': '⚠️ they hold it', 'Typically disclosed': 'no'},
    {'Disclosure level': 'Latent computation',            'We demonstrated': '❌',
     'Vendor could disclose': '❌ impossible', 'Typically disclosed': 'n/a'},
])
ladder

**Only the last row is a technical limit.** Everything above it is a choice.

And even our ceiling is not omniscience: we **verify** the context we built, **trust**
the vendor's usage counts, and can only **assert** the size of reasoning we cannot see.
The residual row is where verified and trusted meet — and where disagreement between
them becomes visible.

## 10. Save results

In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

df_runs = pd.DataFrame([{
    'ticket': r.ticket_id,
    'difficulty': next(x for x in corpus.tickets if x.id == r.ticket_id).difficulty,
    'llm_calls': len(r.all_calls), 'tool_calls': len(r.tool_calls),
    'input_tokens': r.total_usage().input_tokens,
    'output_tokens': r.total_usage().output_tokens,
    'reasoning_tokens': r.total_usage().reasoning_tokens,
    'cached_tokens': r.total_usage().cached_tokens,
    'cost_usd': round(r.total_cost, 6), 'latency_ms': round(r.latency_ms),
    'cited': ', '.join(r.cited_articles), 'escalated': r.escalated,
    'passed': next(o.passed for o in outcomes if o.ticket_id == r.ticket_id),
} for r in runs])
df_runs.to_csv(Config.OUTPUT_DIR / f'support_runs_{ts}.csv', index=False)

attribution.attribution_dataframe(tracer).to_csv(
    Config.OUTPUT_DIR / f'token_attribution_{ts}.csv', index=False)
path = tracer.save_all_traces(Config.TRACE_DIR, timestamp=f'support_{ts}')

print(f'saved → {Config.OUTPUT_DIR}')
print(f'OTLP traces → {path}')
print(f'\ntotal spend for this run: ${df_runs.cost_usd.sum():.4f} '
      f'({df_runs.input_tokens.sum() + df_runs.output_tokens.sum():,} tokens)')
df_runs